# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an interactive walkthrough for loading, exploring, and processing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is provided via a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The dataset covers socio-demographic characteristics, rangeland management knowledge, and ordered logistic regression outputs from 475 households in Northern Kenya.

In [ ]:
# Ensure mlcroissant library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We use `mlcroissant` to load metadata and records from the FAIR² dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')} | Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview

Let's review available record sets (tables) and their structure using their Croissant `@id`. We'll also preview the available fields in each.

In [ ]:
# List all Record Sets by @id
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
else:
    print("No top-level record sets found in metadata. Attempting to automatically detect record sets from manifest...")
    # Advanced: Detect record sets if not present at top level.
    # mlcroissant exposes record_sets from the dataset object itself via dataset.record_sets
    record_sets = [r['@id'] for r in dataset.record_sets]

for idx, rs_id in enumerate(record_sets, start=1):
    print(f"{idx}. Record Set @id: {rs_id}")
    # Get fields of record set:
    rs = next((r for r in dataset.record_sets if r['@id'] == rs_id), None)
    if rs:
        fields = rs.get('field', [])
        if isinstance(fields, dict):  # Single field
            fields = [fields]
        print(f"   Fields (@id): {[f['@id'] if type(f)==dict and '@id' in f else f for f in fields]}")

## 3. Data Extraction

For each record set, we'll extract all records into Pandas DataFrames.

***Note:*** All data references below use Croissant `@id` values.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

for rs_id in record_sets:
    print(f"Loading records from record set: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"- Columns: {df.columns.tolist()}")
        print(f"- Example records:")
        display(df.head(2))
    except Exception as e:
        print(f"Failed to load record set {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

In this section, select a *numeric* field (by Croissant `@id`) and perform operations such as outlier filtering, normalization, and grouping.

*Modify the variables below as needed for your exploration.*

In [ ]:
# Example: select record set and numeric field for EDA (update these with your actual record set and field @ids)
# For demo, let's auto-select the first record set with at least one numeric field

selected_rs_id = None
numeric_field_id = None
group_field_id = None

# Identify candidate numeric columns
for rs_id, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_cols:
        selected_rs_id = rs_id
        numeric_field_id = numeric_cols[0]  # Pick the first numeric column
        # Optionally choose a group by field if possible
        candidate_group_fields = [col for col in df.columns if df[col].dtype == object and col not in [numeric_field_id]]
        if candidate_group_fields:
            group_field_id = candidate_group_fields[0]
        break

if selected_rs_id is None or numeric_field_id is None:
    print("No suitable record set or numeric field found for EDA. Please inspect earlier outputs and set these variables manually.")
else:
    print(f"Selected record set @id: {selected_rs_id}")
    print(f"Numeric field (column): {numeric_field_id}")
    if group_field_id:
        print(f"Grouping by field (column): {group_field_id}")

    df = dataframes[selected_rs_id]
    # Remove null values
    filtered_df = df[df[numeric_field_id].notnull()]
    # Filter rows for outliers (e.g., values greater than the 90th percentile)
    threshold = filtered_df[numeric_field_id].quantile(0.9)
    extreme_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Records with {numeric_field_id} > {threshold:.3f} (90th percentile):")
    display(extreme_df[[numeric_field_id]].head())

    # Normalization (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized '{numeric_field_id}' (z-score):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouped aggregation
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization

Let's visualize distributions and group comparisons within the dataset.

In [ ]:
import matplotlib.pyplot as plt

if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id]
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=30, color='cornflowerblue', alpha=0.7)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.grid(False)
    plt.show()

    # If group field available, make a boxplot
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df[[group_field_id, numeric_field_id]].boxplot(by=group_field_id, column=numeric_field_id, grid=False)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}' group")
        plt.suptitle("")  # Suppress default title
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to access, process, and visualize data from a FAIR² dataset package using the [mlcroissant](https://github.com/mlcommons/croissant) API.

- Croissant `@id` values enable robust, traceable data access for fields, tables, and all entities.
- The dataset can be explored interactively in Pandas, enabling custom filtering, normalization, and aggregation.
- For further analysis, consult the complete field list using each record set's schema and leverage domain knowledge about data provenance and variable semantics.

_For questions or more complex queries, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) or the dataset's detailed Croissant schema._